In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

PROCESSED = Path("../data/processed")

precios = pd.read_parquet(PROCESSED / "precios_secciones_2024.parquet")    # tabla normal
secciones = gpd.read_parquet(PROCESSED / "secciones.parquet")             # formas de las secciones
pois = gpd.read_parquet(PROCESSED / "pois_combinados.parquet")

# Parques y monumentos no están en pois_combinados (los descargó preparar_lugares.py): los añadimos
RAW_OSM = Path("../data/raw/osm")
extra = pd.concat([gpd.read_parquet(RAW_OSM / f"{c}.parquet") for c in ["parques", "monumentos"]], ignore_index=True)
pois = gpd.GeoDataFrame(pd.concat([pois, extra], ignore_index=True), geometry="geometry", crs=pois.crs)
barrios = gpd.read_parquet(PROCESSED / "barrios_indices.parquet")

print("--- Precios por sección ---")
print(precios.shape)
print(precios.dtypes.to_string())
print(precios.head(3).T)

print("\n--- Secciones ---")
print(secciones.shape, secciones.crs)
print(list(secciones.columns))

print("\n--- Columnas de barrios_indices ---")
print(list(barrios.columns))

print("\n--- Lugares ---")
print(len(pois), "·", pois["categoria"].nunique(), "categorías")

In [3]:
datos = secciones.merge(precios, on="cod_seccion_ine", how="inner")
print("Secciones con precio:", len(datos))

# Con pocas viviendas alquiladas, el precio medio es poco fiable (igual que el filtro de 300 por barrio)
MIN_VIVIENDAS_SECCION = 20
datos = datos[datos["n_viviendas"] >= MIN_VIVIENDAS_SECCION].copy()
print(f"Con al menos {MIN_VIVIENDAS_SECCION} viviendas:", len(datos))

# Un punto dentro de cada sección: desde él mediremos distancias y contaremos lugares
puntos = gpd.GeoSeries(datos.representative_point().values, index=datos["cod_seccion_ine"], crs=secciones.crs)
tabla = datos.set_index("cod_seccion_ine")[["cod_barrio", "cod_distrito", "eur_m2", "n_viviendas", "m2"]]
tabla.describe()

Secciones con precio: 2408
Con al menos 20 viviendas: 2345


,eur_m2,n_viviendas,m2
count,2345.000000,2345.000000,2345.000000
mean,13.687373,126.794030,71.579531
std,3.034531,70.935363,19.441097
min,7.211851,20.000000,31.000000
25%,11.320755,77.000000,58.000000
50%,13.042553,114.000000,67.000000
75%,16.031469,161.000000,81.000000
max,25.726073,630.000000,190.000000


In [ ]:
RADIO = 500  # metros: unos 7-8 minutos andando

# 1) Cuántos lugares de cada tipo hay a menos de 500 m
#    (guardamos el código en una columna "seccion" para no depender de cómo la llame sjoin)
zonas = gpd.GeoDataFrame({"seccion": puntos.index}, geometry=puntos.buffer(RADIO).values, crs=puntos.crs)
cruce = gpd.sjoin(pois[["categoria", "geometry"]], zonas, predicate="within")
conteos = (
    cruce.groupby(["seccion", "categoria"]).size()
    .unstack(fill_value=0)
    .add_prefix("n500_")
    .reindex(tabla.index, fill_value=0)
)

# 2) Distancia al más cercano de cada tipo importante (en metros)
puntos_gdf = gpd.GeoDataFrame(geometry=puntos)
def distancia_al_mas_cercano(categorias):
    destinos = pois.loc[pois["categoria"].isin(categorias), ["geometry"]]
    r = gpd.sjoin_nearest(puntos_gdf, destinos, distance_col="d")
    return r.groupby(level=0)["d"].first()

distancias = pd.DataFrame({
    "dist_tren_m": distancia_al_mas_cercano(["estaciones_metro", "estaciones_cercanias", "estaciones_metro_ligero"]),
    "dist_bus_m": distancia_al_mas_cercano(["paradas_bus"]),
    "dist_parque_m": distancia_al_mas_cercano(["parques"]),
})

# 3) Centralidad: distancia en línea recta a la Puerta del Sol
sol = gpd.GeoSeries.from_xy([-3.7035], [40.4169], crs=4326).to_crs(puntos.crs).iloc[0]
centralidad = (puntos.distance(sol) / 1000).rename("dist_sol_km")

# 4) Transporte real (r5py, por barrio): destinos a menos de 30 min y minutos hasta Sol
tiempos = pd.read_parquet(PROCESSED / "tiempos_barrio.parquet")
rejilla = gpd.read_parquet(PROCESSED / "rejilla_destinos.parquet").to_crs(puntos.crs)
id_sol = rejilla.loc[rejilla.distance(sol).idxmin(), "id"]
por_barrio = pd.DataFrame({
    "destinos_30min": tiempos[tiempos["minutos"] <= 30].groupby("cod_barrio").size(),
    "min_tp_sol": tiempos[tiempos["id_destino"] == id_sol].set_index("cod_barrio")["minutos"],
})
transporte = tabla[["cod_barrio"]].join(por_barrio, on="cod_barrio").drop(columns="cod_barrio")

# Todo junto
X = pd.concat([conteos, distancias, centralidad, transporte, tabla[["m2"]].rename(columns={"m2": "m2_medio"})], axis=1)
y = tabla["eur_m2"]
grupos = tabla["cod_distrito"]   # para la validación por distritos

print(X.shape)
print("\n--- Correlación de cada variable con el €/m² (las 15 más fuertes) ---")
print(X.corrwith(y).sort_values(key=abs, ascending=False).head(15).round(2).to_string())

print("\n--- Huecos por columna (debería salir vacío o casi) ---")
print(X.isna().sum()[lambda s: s > 0])

pd.concat([X, y, tabla[["cod_barrio", "cod_distrito", "n_viviendas"]]], axis=1).to_parquet(PROCESSED / "modelo_datos.parquet")

In [6]:
from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Algunos barrios no tienen tiempo hasta Sol (más de 90 min): rellenamos con la mediana
X = X.fillna(X.median())

modelos = {
    "Media de Madrid (referencia)": DummyRegressor(strategy="mean"),
    "Regresión lineal (Ridge)": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20))),
    "LightGBM": LGBMRegressor(
        n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=20,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=42, verbose=-1,
    ),
}

validacion_espacial = LeaveOneGroupOut()   # entrena con 20 distritos y predice el que falta
predicciones = {}
filas = []
for nombre, modelo in modelos.items():
    pred = cross_val_predict(modelo, X, y, groups=grupos, cv=validacion_espacial)
    predicciones[nombre] = pred
    filas.append({
        "modelo": nombre,
        "MAE (€/m²)": mean_absolute_error(y, pred),
        "error medio (%)": mean_absolute_error(y, pred) / y.mean() * 100,
        "error en un piso de 70 m² (€/mes)": mean_absolute_error(y, pred) * 70,
        "R²": r2_score(y, pred),
    })

# La "trampa": LightGBM evaluado mezclando secciones al azar (sin separar por distritos)
pred_azar = cross_val_predict(modelos["LightGBM"], X, y, cv=KFold(5, shuffle=True, random_state=42))
filas.append({
    "modelo": "LightGBM con validación al azar (optimista)",
    "MAE (€/m²)": mean_absolute_error(y, pred_azar),
    "error medio (%)": mean_absolute_error(y, pred_azar) / y.mean() * 100,
    "error en un piso de 70 m² (€/mes)": mean_absolute_error(y, pred_azar) * 70,
    "R²": r2_score(y, pred_azar),
})

resultados = pd.DataFrame(filas).round(2)
resultados

c:\Users\Usuario\miniforge3\envs\madrid\Lib\site-packages\sklearn\utils\extmath.py:1211: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
c:\Users\Usuario\miniforge3\envs\madrid\Lib\site-packages\sklearn\utils\extmath.py:1216: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
c:\Users\Usuario\miniforge3\envs\madrid\Lib\site-packages\sklearn\utils\extmath.py:1240: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


ValueError: Input X contains NaN.
_RidgeGCV does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values